In [3]:
import matplotlib.pyplot as plt
import numpy as np
import sys
import torch
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
import os, math
import numpy as np
import torch
import matplotlib.pyplot as plt
import src
from src.train.regressor_train_step import get_root_dataset, _predict_on_loader, _collect_all_splits


def _auto_layout(n, layout=None):
    """根据数量自动给出 (rows, cols)。支持手动覆盖。"""
    if layout is not None:
        return layout
    if n == 2:  return (1, 2)
    if n == 4:  return (2, 2)
    if n == 6:  return (2, 3)
    cols = min(3, n)
    rows = math.ceil(n / cols)
    return rows, cols


def plot_models_series(models, model_names, loaders, device, save_path,
                       which="Test", concat_splits=False, layout=None, dpi=120):
    """
    models / model_names: 数量可为 2、4、6（或更多）
    which: 只看某个划分；concat_splits=True 时拼接 Train→Val→Test
    layout: (rows, cols) 可选；不传则自动 2→1×2, 4→2×2, 6→2×3
    """
    assert len(models) == len(model_names) >= 2
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

    per_model_data = {n: _collect_all_splits(m, loaders, device)
                      for m, n in zip(models, model_names)}

    r, c = _auto_layout(len(models), layout)
    fig, axs = plt.subplots(r, c, figsize=(6*c, 4*r), squeeze=False)
    axs = axs.ravel()

    for ax, name in zip(axs, model_names):
        d = per_model_data[name]
        if concat_splits:
            idx = 0
            for split in ("Train", "Validation", "Test"):
                y_t, y_p = d[split]
                n = len(y_t); rng = np.arange(idx, idx+n)
                ax.plot(rng, y_t, label=f"{split} - True")
                ax.plot(rng, y_p, linestyle="--", label=f"{split} - Pred")
                idx += n
            ax.set_xlabel("Sample Index (Train→Val→Test)")
        else:
            y_t, y_p = d[which]
            rng = np.arange(len(y_t))
            ax.plot(rng, y_t, label="True")
            ax.plot(rng, y_p, linestyle="--", label="Pred")
            ax.set_xlabel(f"Sample Index ({which})")
        ax.set_title(name); ax.set_ylabel("Magnitude"); ax.grid(True)

    for ax in axs[len(models):]:
        ax.axis("off")

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(6, len(labels)),
               bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(save_path, dpi=dpi); plt.show()

# ---------- 通用函数：多模型散点图 ----------
def plot_models_scatter(models, model_names, loaders, device, save_path,
                        which="Test", layout=None, dpi=120):
    """
    每个子图画 y_true vs y_pred + y=x 参考线；支持 2 / 4 / 6 / 更多
    """
    assert len(models) == len(model_names) >= 2
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

    per_model_data = {n: _collect_all_splits(m, loaders, device)
                      for m, n in zip(models, model_names)}

    r, c = _auto_layout(len(models), layout)
    fig, axs = plt.subplots(r, c, figsize=(5*c, 4*r), squeeze=False)
    axs = axs.ravel()

    for ax, name in zip(axs, model_names):
        y_t, y_p = per_model_data[name][which]
        ax.scatter(y_t, y_p, s=10, alpha=0.6)
        mn, mx = np.min([y_t, y_p]), np.max([y_t, y_p])
        ax.plot([mn, mx], [mn, mx], linestyle="--")
        ax.set_title(name); ax.set_xlabel(f"True ({which})"); ax.set_ylabel("Pred"); ax.grid(True)

    for ax in axs[len(models):]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=dpi); plt.show()


In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
from src.models.builders import ModelBuilder
from src.train.config_setup import load_args_from_checkpoint,load_model_from_checkpoint
checkpoint_dir = Path("../checkpoints/clf_mixer_attnpl_t_20250905-184209") 
checkpoint_path = checkpoint_dir / "best_model_1.pth"
check_point = torch.load(checkpoint_path,weights_only=False)
args = load_args_from_checkpoint(None, check_point)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_builder = ModelBuilder.by_name(args.model.lower())()
model = model_builder(args, device)
model,_,_ = load_model_from_checkpoint(model, check_point)
model = torch.compile(model)
model.eval() 

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.